# Diamonds – цена бриллианта: как выбросы ломают MAPE

**Задача.** Предсказать `total_sales_price` (USD) по каратам, грейдам (color/clarity/cut/symmetry/polish),
процентам depth/table и трём замерам в мм. Метрика – **MAPE**. Групповой лучший результат ≈ **7.31%**.

**Ошибка, которую разбираем.** В исходном ноутбуке отсечка выбросов делалась циклом `1.5·IQR` по
**всем** числовым колонкам, **включая целевую** `total_sales_price`. Это удаляет из обучения весь дорогой
хвост, тогда как настоящий `test.csv` держит полный диапазон цен → модель не дотягивается до дорогих
камней → **MAPE ≈ 18.7** (а если ту же чистку сделать ещё и до сплита, «почистится» и тест → оптимистичные
~11.7, которые на реальном тесте всё равно превращаются в ~18.7).

**Правило.** СНАЧАЛА сплит, выбросы ищем ТОЛЬКО на train, по **таргету не фильтруем**, тестовую выборку
руками не трогаем. Ниже – один и тот же градиентный бустинг, разница только в работе с выбросами.

> Продакшн-версия этого пайплайна – скрипт `backend/training/diamonds.py` (он и пишет
> `backend/models/diamonds.joblib`). Исходные учебные ноутбуки: `~/personal/ML-projects/Diamonds/`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

DATA = "../../data/diamonds"
train = pd.read_csv(f"{DATA}/train.csv").drop_duplicates().reset_index(drop=True)
X_test = pd.read_csv(f"{DATA}/test.csv")
y_test = pd.read_csv(f"{DATA}/test_Y_true.csv")["total_sales_price"].to_numpy(float)

TARGET = "total_sales_price"
NUM = ["size", "depth_percent", "table_percent", "meas_length", "meas_width", "meas_depth"]
GRADE_ORDER = {
    "color":    ["D","E","F","G","H","I","J","K","L","M"],
    "clarity":  ["IF","VVS1","VVS2","VS1","VS2","SI1","SI2","I1","I2","I3"],
    "cut":      ["Excellent","Very Good"],
    "symmetry": ["Excellent","Very Good"],
    "polish":   ["Excellent","Very Good"],
}
CAT = list(GRADE_ORDER)
FEATURES = NUM[:1] + CAT + NUM[1:]  # size, грейды, проценты, замеры

def mape(y, pred):
    return float(np.mean(np.abs((y - np.clip(pred, 1, None)) / y)))

print(f"train: {len(train)} строк | test: {len(X_test)} строк | цена train {train[TARGET].min()}..{train[TARGET].max()}")

train: 67406 строк | test: 22453 строк | цена train 242..19996


## Один пайплайн для обоих сценариев

Модель одна и та же (`HistGradientBoostingRegressor`, `loss='gamma'` – он оптимизирует относительную
ошибку под MAPE). Категориальные грейды кодируем `OrdinalEncoder` по порядку качества; невозможные нули
в числовых фичах чиним медианой train **внутри** пайплайна (fit только на train → без утечки).

In [ ]:
def make_model():
    numeric = Pipeline([
        ("zeros", SimpleImputer(missing_values=0, strategy="median")),  # замеры/проценты == 0 → медиана
        ("imputer", SimpleImputer(strategy="median")),
    ])
    preproc = ColumnTransformer([
        ("num", numeric, NUM),
        ("cat", OrdinalEncoder(categories=[GRADE_ORDER[c] for c in CAT],
                               handle_unknown="use_encoded_value", unknown_value=-1), CAT),
    ])
    hgb = HistGradientBoostingRegressor(loss="gamma", max_iter=600, learning_rate=0.05,
                                        max_leaf_nodes=63, min_samples_leaf=20, l2_regularization=0.1,
                                        early_stopping=True, random_state=42)
    return Pipeline([("preproc", preproc), ("model", hgb)])

## ❌ Неправильно: IQR по всем числовым колонкам, включая таргет

Точь-в-точь как в исходном ноутбуке. Смотрим, сколько строк удаляется, затем учим модель на «почищенном»
train и меряем на настоящем тесте.

In [ ]:
bad = train.copy()
before = len(bad)
for col in bad.select_dtypes(include=["float64", "int64"]).columns:  # включая total_sales_price!
    q1, q3 = bad[col].quantile(0.25), bad[col].quantile(0.75)
    iqr = q3 - q1
    bad = bad[(bad[col] >= q1 - 1.5 * iqr) & (bad[col] <= q3 + 1.5 * iqr)]
print(f"осталось {len(bad)} из {before} строк (удалено {before - len(bad)}, {(1 - len(bad)/before):.0%})")
print(f"макс. цена в train ПОСЛЕ отсечки: {bad[TARGET].max()}  (в тесте есть до {int(y_test.max())})")

m_bad = make_model().fit(bad[FEATURES], bad[TARGET])
mape_bad = mape(y_test, m_bad.predict(X_test[FEATURES]))
print(f"MAPE на настоящем test.csv: {mape_bad:.4f}  →  {mape_bad*100:.2f}%")

осталось 53500 из 67406 строк (удалено 13906, 21%)
макс. цена в train ПОСЛЕ отсечки: 3927  (в тесте есть до 19968)
MAPE на настоящем test.csv: 0.1439  →  14.39%


Модель обучена на усечённом диапазоне (дорогой хвост вырезан вместе с крупными каратами), а тест держит
полный диапазон → относительная ошибка на дорогих камнях огромная. Дело не в модели, а в том, что мы
испортили обучающее распределение, тронув таргет.

## ✅ Правильно: сплит первым, чистка только на train, таргет не трогаем

Никакой фильтрации по таргету. Сплит делаем сразу, вся чистка/импутация – внутри пайплайна (fit на train).
Сначала честная внутренняя валидация, затем финальная модель на полном train и скоринг на настоящем тесте.

In [ ]:
x_tr, x_val, y_tr, y_val = train_test_split(train[FEATURES], train[TARGET], test_size=0.2, random_state=42)
m_val = make_model().fit(x_tr, y_tr)
print(f"внутренний holdout MAPE: {mape(y_val.to_numpy(float), m_val.predict(x_val)):.4f}")

m_ok = make_model().fit(train[FEATURES], train[TARGET])   # финальная модель на всём train
mape_ok = mape(y_test, m_ok.predict(X_test[FEATURES]))
print(f"MAPE на настоящем test.csv: {mape_ok:.4f}  →  {mape_ok*100:.2f}%")

внутренний holdout MAPE: 0.0692
MAPE на настоящем test.csv: 0.0688  →  6.88%


## Итог

| Подход | Что не так | MAPE на `test.csv` |
|---|---|---|
| IQR по всем колонкам, включая таргет (**тот же** бустинг) | из train вырезан дорогой хвост | **≈14.4%** |
| Исходный ноутбук (та же ошибка + KNN) | выбросы **и** слабая модель | 18.68% |
| Групповой лучший | – | 7.31% |
| **Сплит → чистка только на train, таргет не трогаем** | – | **≈6.9%** |

Вывод: даже на сильной модели некорректная отсечка по таргету **удваивает** MAPE (6.9% → 14.4%); в
исходном ноутбуке к этому добавился слабый KNN, отсюда 18.68%. Достаточно не фильтровать по таргету и
держать чистку внутри пайплайна на train – и тот же бустинг обгоняет групповой лучший результат.
Продакшн-скрипт: `backend/training/diamonds.py`.